### Exercise 2: remote CSV and Parquet tools with Polars

In [2]:
import io
import json
import os
from typing import Callable
from urllib.parse import urlparse

import polars as pl
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
CSV_URL = (
    "https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/"
    "refs/heads/master/outputs/dataset_final.csv"
)
PARQUET_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-01.parquet"
)

print("Configuration loaded.")
print("Gemini model:", MODEL)

Configuration loaded.
Gemini model: gemini-3.5-flash


In [3]:
MAX_ROWS = 50
MAX_DOWNLOAD_BYTES = 150 * 1024 * 1024


def _validate_url(url: str) -> None:
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Only valid HTTP(S) URLs are supported.")


def _download(url: str) -> bytes:
    _validate_url(url)
    response = requests.get(url, timeout=90)
    response.raise_for_status()

    content_length = response.headers.get("content-length")
    if content_length and int(content_length) > MAX_DOWNLOAD_BYTES:
        raise ValueError("Remote file is larger than 150 MB.")
    if len(response.content) > MAX_DOWNLOAD_BYTES:
        raise ValueError("Downloaded file is larger than 150 MB.")

    return response.content


def _dataframe_to_text(df: pl.DataFrame, source_url: str) -> str:
    schema = {name: str(dtype) for name, dtype in df.schema.items()}
    return (
        f"Source URL: {source_url}\n"
        f"Returned sample shape: {df.shape}\n"
        f"Schema: {json.dumps(schema)}\n"
        "Rows in CSV format:\n"
        f"{df.write_csv()}"
    )

In [4]:
def read_remote_csv(url: str, max_rows: int = 30) -> str:
    """Read a limited sample of a remote CSV file and return it as text."""
    row_limit = max(1, min(int(max_rows), MAX_ROWS))
    data = _download(url)
    df = pl.read_csv(io.BytesIO(data), n_rows=row_limit)
    return _dataframe_to_text(df, url)


def read_remote_parquet(url: str, max_rows: int = 30) -> str:
    """Read a limited sample of a remote Parquet file and return it as text."""
    row_limit = max(1, min(int(max_rows), MAX_ROWS))
    data = _download(url)
    df = pl.read_parquet(io.BytesIO(data), n_rows=row_limit)
    return _dataframe_to_text(df, url)

### Test the tools without Gemini

In [5]:
csv_preview = read_remote_csv(CSV_URL, max_rows=5)
print(csv_preview[:3000])

Source URL: https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv
Returned sample shape: (5, 13)
Schema: {"name": "String", "CID": "Int64", "CAS": "String", "SMILES": "String", "source": "String", "year": "Int64", "toxicity_type": "String", "herbicide": "Int64", "fungicide": "Int64", "insecticide": "Int64", "other_agrochemical": "Int64", "label": "Int64", "ppdb_level": "Int64"}
Rows in CSV format:
name,CID,CAS,SMILES,source,year,toxicity_type,herbicide,fungicide,insecticide,other_agrochemical,label,ppdb_level
Ethanedioic acid,971,144-62-7,O=C(O)C(=O)O,ECOTOX,1832,Contact,0,0,0,0,0,0
Para-cymene,7463,99-87-6,Cc1ccc(C(C)C)cc1,BPDB,1833,Other,0,0,0,1,0,1
Kieselguhr,24261,61790-53-2,O=[Si]=O,ECOTOX,1833,Contact,0,0,0,0,0,0
Benzoic acid,243,65-85-0,O=C(O)c1ccccc1,ECOTOX,1833,Contact,1,1,1,0,0,1
Tetradifon (Ref: ENT 23737),8305,116-29-0,O=S(=O)(c1ccc(Cl)cc1)c1cc(Cl)c(Cl)cc1Cl,PPDB,1836,Oral,0,0,0,1,1,1



In [6]:
parquet_preview = read_remote_parquet(PARQUET_URL, max_rows=5)
print(parquet_preview[:3000])

Source URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Returned sample shape: (5, 19)
Schema: {"VendorID": "Int32", "tpep_pickup_datetime": "Datetime(time_unit='ns', time_zone=None)", "tpep_dropoff_datetime": "Datetime(time_unit='ns', time_zone=None)", "passenger_count": "Int64", "trip_distance": "Float64", "RatecodeID": "Int64", "store_and_fwd_flag": "String", "PULocationID": "Int32", "DOLocationID": "Int32", "payment_type": "Int64", "fare_amount": "Float64", "extra": "Float64", "mta_tax": "Float64", "tip_amount": "Float64", "tolls_amount": "Float64", "improvement_surcharge": "Float64", "total_amount": "Float64", "congestion_surcharge": "Float64", "Airport_fee": "Float64"}
Rows in CSV format:
VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Air

In [14]:
def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    common_parameters = {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "Public HTTP(S) URL of the dataset file.",
            },
            "max_rows": {
                "type": "integer",
                "description": "Number of rows to read, from 1 to 50.",
                "minimum": 1,
                "maximum": 50,
                "default": 30,
            },
        },
        "required": ["url"],
    }

    definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": (
                    "Read a sample from a public CSV URL with Polars and return "
                    "its schema and rows as text."
                ),
                "parameters": common_parameters,
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": (
                    "Read a sample from a public Parquet URL with Polars and return "
                    "its schema and rows as text."
                ),
                "parameters": common_parameters,
            },
        },
    ]

    functions = {
        "read_remote_csv": read_remote_csv,
        "read_remote_parquet": read_remote_parquet,
    }
    return definitions, functions

In [13]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=GEMINI_API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        timeout=90.0,
        max_retries=1,
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a data analysis assistant. Always use the matching tool "
                "when the user provides a CSV or Parquet URL. Base your answer only "
                "on the returned sample and clearly state that conclusions concern "
                "the sampled rows, not necessarily the complete dataset."
            ),
        },
        {"role": "user", "content": prompt},
    ]
    tool_definitions, tool_name_to_func = get_tool_definitions()

    print(f"PROMPT:\n{prompt}\n")
    for step in range(1, 11):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_definitions,
            tool_choice="auto",
            max_completion_tokens=1500,
            reasoning_effort="low",
        )
        resp_message = response.choices[0].message
        message_data = resp_message.model_dump(exclude_none=True)
        messages.append(message_data)

        print(f"GENERATED MESSAGE {step}:")
        print(json.dumps(message_data, indent=2))
        print()

        if not resp_message.tool_calls:
            return resp_message.content or ""

        for tool_call in resp_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            func_result = tool_name_to_func[func_name](**func_args)

            print("TOOL CALL:")
            print(json.dumps({"name": func_name, "arguments": func_args}, indent=2))
            print("TOOL RESULT PREVIEW:")
            print(func_result[:4000])
            print()

            messages.append(
                {
                    "role": "tool",
                    "content": func_result,
                    "tool_call_id": tool_call.id,
                }
            )

    return f"Could not resolve request, last response: {resp_message.content}"

## CSV test with ApisTox

In [15]:
csv_prompt = f"""
Read this CSV dataset using the CSV tool:
{CSV_URL}

Use 30 rows. What columns are present? Summarize the sampled records and identify
any columns that appear useful for analyzing chemical toxicity. Do not claim that
the sample represents the entire dataset.
""".strip()

csv_answer = make_llm_request(csv_prompt)
print("FINAL CSV RESPONSE:")
print(csv_answer)

PROMPT:
Read this CSV dataset using the CSV tool:
https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv

Use 30 rows. What columns are present? Summarize the sampled records and identify
any columns that appear useful for analyzing chemical toxicity. Do not claim that
the sample represents the entire dataset.

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "1io6ucjy",
      "function": {
        "arguments": "{\"max_rows\":30,\"url\":\"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv\"}",
        "name": "read_remote_csv"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EpoCCpcCAQw51sdD+4FC/RJuYLa7nfbzX2dyQBZJvoKsUrA1/tdNvXuqbogv5GAZbhEYOwMNXRA00Jr10UcUX46H+g+Eih6noVo9YT1mT/RMKswYdxTVFkip39GkiNFL6qF5YBFQIHtZI3v2fj8ZHEqiXwYDh9UQNDuL9iTk0EyF8heHujaCU8j05douV9whGzfMqk6u8MN3CZgQJ8+e24OW5u

### Parquet test with NYC yellow taxi trips

In [16]:
parquet_prompt = f"""
Read this Parquet dataset using the Parquet tool:
{PARQUET_URL}

Use 30 rows. Based only on the returned sample:
1. What columns are available?
2. What are the approximate average trip distance and average total amount?
3. What caveat should be stated about using only 30 rows?
""".strip()

parquet_answer = make_llm_request(parquet_prompt)
print("FINAL PARQUET RESPONSE:")
print(parquet_answer)

PROMPT:
Read this Parquet dataset using the Parquet tool:
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet

Use 30 rows. Based only on the returned sample:
1. What columns are available?
2. What are the approximate average trip distance and average total amount?
3. What caveat should be stated about using only 30 rows?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "bhdx0ldu",
      "function": {
        "arguments": "{\"max_rows\":30,\"url\":\"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet\"}",
        "name": "read_remote_parquet"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "Es0DCsoDAQw51scsdt0U2GGBMxrkyiXppB31tG8j99408Q6aluXzY1Ygf5VhBHibKijXgWUazsK1O7IYfe3SLlomHuWFsWxl4m26VsXmV9kqjsdfWb+PJ+QswUppBvj0SvYlnk+JpfGPri57UPWXVQsrckRjwf/98bVb6rTf/X5xCpsyuett2QHDdH2aItq/5toDlxIm2STGXw7G1zRAQplBGs5VWiNBw7zTxQaYrEY23vIvpMT9FeS2w

### Addiotional questions

In [17]:
additional_prompt = f"""
Use the CSV tool to read 20 rows from:
{CSV_URL}

Which columns in the sample look categorical, numerical, or textual?
""".strip()

additional_answer = make_llm_request(additional_prompt)
print(additional_answer)

PROMPT:
Use the CSV tool to read 20 rows from:
https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv

Which columns in the sample look categorical, numerical, or textual?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "g3sdoxqp",
      "function": {
        "arguments": "{\"url\":\"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv\",\"max_rows\":20}",
        "name": "read_remote_csv"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EqECCp4CAQw51sfzCwhuDsYhceM+NP45gL2lbtLrWPwFwtHRycgl3aKw31QC12omz1KO/tBCC1ZZ6ph07f0wh+RDKYVsKhSGRiopT6PzNgNAn+pDuaC0i/jvNcEmkkDCmxlqb7I/G49s5thgi9PappKKvDISTdP9yvsXFviaiWAygC+k0WxrlvBFMMEp0jKjnQiYTSjc0WHI0sLOSxONu2bX5M1sUJlE8TMnxGuAfHF9tZ4birprJuI7Dcu9mYILNMOXycV0LBvBhFigKjJeXLmHytnS9kDqV7acCFzK/f60ACs1BF0PYIGsSnqEB7v5TMKQgwKQbzjQTN2ewTKNJ5TMctSQsJUfQHfO7B

In [18]:
additional_prompt = f"""
Use the CSV tool to read 20 rows from:
{CSV_URL}

What are the minimum and maximum numeric values visible in the sample?
""".strip()

additional_answer = make_llm_request(additional_prompt)
print(additional_answer)

PROMPT:
Use the CSV tool to read 20 rows from:
https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv

What are the minimum and maximum numeric values visible in the sample?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "s9433z6c",
      "function": {
        "arguments": "{\"max_rows\":20,\"url\":\"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv\"}",
        "name": "read_remote_csv"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EpkCCpYCAQw51sdizGTieYYkpKpiFi1JDq1QH76LK1oZ6SIpAYVham4iIcDmdvfsoH9gMa+Hndm60MFqYdY+DTgBStxoEhFIcHl+jMwHQ5PSGQrBOgGdZLuCR/zcOQEA71o5pBPqn5F7SjU9DAj3pcNFHZa+Dvdo7ceS15TARQWL1oyklz290mlaO/1e5Dl95amwpvV0F84v55De4ka/x0aalOOm9mExyitbgv/gFf/4cSDelEq7R36/3Bc38S2d+nBOA4Pb57M4+u0S9NMzzK0+fx3fdKUef1oe/I8I3far3xfk0EL9UQn9lk1grE12XB3vYFB2r4yGW6BXDMpuYq82bZJNfmstJUzH

In [ ]:
additional_prompt = f"""
Use the CSV tool to read 20 rows from:
{CSV_URL}

Which columns would be useful for a predictive model and why?
""".strip()

additional_answer = make_llm_request(additional_prompt)
print(additional_answer)

PROMPT:
Use the CSV tool to read 20 rows from:
https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv

Which columns would be useful for a predictive model and why?



### Results and conclusions

In this exercise, I implemented two tools for reading remote datasets: `read_remote_csv` and `read_remote_parquet`. Both tools downloaded the file from the URL, opened it with Polars, limited the number of rows, and returned the schema and data as text for Gemini.

For the ApisTox dataset, Gemini correctly selected `read_remote_csv` and requested 30 rows. The returned sample had 30 rows and 13 columns. Gemini identified columns such as `name`, `CID`, `CAS`, `SMILES`, `toxicity_type`, `label`, and the agrochemical flags. It suggested that `SMILES`, `label`, `ppdb_level`, and `toxicity_type` could be useful for toxicity analysis. In the additional tests, it also divided the columns into categorical, numerical, and textual groups. For the first 20 rows, the smallest numeric value was 0 and the largest was 637,566 in the `CID` column.

For the NYC taxi dataset, Gemini correctly selected `read_remote_parquet` and requested 30 rows. The tool returned 30 rows and 19 columns, including trip distance, pickup and drop-off times, fare, tips, and total amount. However, the final Gemini response was cut off while listing the columns, so it did not answer the questions about average trip distance and average total amount. The tool worked correctly, but the model response should be generated again with a larger `max_completion_tokens` value or with a shorter question.

Overall, both tools worked and Gemini selected the correct one based on the file format. Using only a small sample is useful because it reduces the amount of text sent to the model.